# Actor-critic on the continuous mountain car

A port of `actor_critic_mountaincar.py` from
[DuarteA/ActorCriticExample](https://github.com/DuarteA/ActorCriticExample) --
the example the deck links to on slide 19.

The previous notebook ran REINFORCE on this problem. Here a **critic** is
added, and the unit's claim is that this reduces the variance of the policy
gradient. The notebook measures that claim rather than repeating it.

The two networks are

- the **actor** $\pi(u\mid x,\theta)$, the same Gaussian policy as before;
- the **critic** $\hat J(x,\alpha)$, a value-function approximation.

and the quantity passed between them is the **temporal difference** of Unit 11,

$$\delta_k=g(x_k,u_k)+\alpha\,\hat J(x_{k+1},\alpha_{k+1})-\hat J(x_k,\alpha_k),$$

which the critic fits and the actor uses as the signed weight on its gradient:

$$\textbf{critic}\quad \alpha_{k+1}=\alpha_k-\beta\,\delta_k\,\nabla_\alpha\hat J(x_k,\alpha_k), \qquad \textbf{actor}\quad \theta_{k+1}=\theta_k-\gamma\,\delta_k\,\nabla_\theta\log\pi(u_k\mid x_k,\theta_k).$$

Note the symbol collision carried over from the deck: $\alpha$ is the critic's
**weight vector** here, while the discount factor is also written $\alpha$ in
Units 10--12. In the code the discount is `cfg.gamma` and the critic weights
are the critic network's parameters.

::: Requirements :::
`torch` and `gymnasium` (classic control only), plus `policy_gradient.py`
next to this notebook. **Sign convention:** Gymnasium maximises reward, the
course minimises cost; the two differ by $g=-R$.

In [ ]:
import sys, time
import numpy as np
import torch
import matplotlib.pyplot as plt

sys.path.insert(0, ".")
import policy_gradient as pg
from policy_gradient import Config, seed_all, make_env, evaluate, smooth

EPISODES = 1000      # the script's default; 200-300 already shows the behaviour
cfg = Config(episodes=EPISODES)
seed_all(cfg.seed)
env = make_env(cfg, seed=cfg.seed)
state_size, action_size = env.observation_space.shape[0], env.action_space.shape[0]
print(f"{cfg.env_id}: {state_size} state dimensions, {action_size} continuous action")
print(f"actor step size {cfg.lr_actor}, critic step size {cfg.lr_critic}")

## Where the critic comes from

The deck derives it in two steps, and both are worth keeping in view.

First, **baseline subtraction**. Any function $b(x)$ of the state alone can be
subtracted from the Q-factor without changing the gradient,

$$-b(x)\sum_u\nabla_\theta\pi(u\mid x,\theta)=-b(x)\nabla_\theta\Big(\sum_u\pi(u\mid x,\theta)\Big)=-b(x)\nabla_\theta(1)=0,$$

so the estimator stays unbiased while its variance changes. Picking $b$ well
is a variance reduction technique.

Second, **let the baseline be learned**. Take $b(x_k)\to\hat J(x_k,\alpha_k)$,
fit it online, and $Q_\pi-b$ becomes the TD error $\delta_k$. That is the
critic, and it is why the deck says actor-critic *is* baseline subtraction.

So the difference from the previous notebook is not the algorithm's shape but
what multiplies the policy gradient: a whole-episode return $G_k$ there, a
one-step temporal difference $\delta_k$ here.

In [ ]:
agent = pg.ActorCritic(state_size, action_size, cfg)
print("actor :", sum(p.numel() for p in agent.actor.parameters()), "parameters")
print("critic:", sum(p.numel() for p in agent.critic.parameters()), "parameters")

## Training

One update per **time step** now, not per episode -- so this is slower per
episode than REINFORCE, and each episode on this environment runs to the
999-step limit unless the car reaches the flag. Expect this to take a while;
reduce `EPISODES` for a first look.

In [ ]:
t0 = time.perf_counter()
scores, actor_losses, critic_losses, delta_std = pg.train_actor_critic(
    agent, env, cfg, verbose_every=100)
print(f"\n{len(scores)} episodes in {time.perf_counter()-t0:.0f} s")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
ax[0].plot(scores, alpha=0.3, lw=1, color="C0")
sm = smooth(scores, 50)
ax[0].plot(np.arange(len(sm)) + 49, sm, color="C3", lw=2, label="moving average (50)")
ax[0].axhline(0, color="k", ls=":", lw=1)
ax[0].axhline(90, color="g", ls="--", lw=1, label="reaching the goal")
ax[0].set_xlabel("Episode"); ax[0].set_ylabel("Episode return")
ax[0].set_title("Actor-critic"); ax[0].legend(fontsize=8); ax[0].grid(True, alpha=0.3)

ax[1].plot(critic_losses, lw=1, color="C2")
ax[1].set_yscale("log"); ax[1].set_xlabel("Episode")
ax[1].set_ylabel("mean critic loss"); ax[1].set_title("Critic: is it learning?")
ax[1].grid(True, alpha=0.3)

ax[2].plot(delta_std, lw=1, color="C4")
ax[2].set_xlabel("Episode"); ax[2].set_ylabel(r"std of $\delta_k$ within an episode")
ax[2].set_title(r"Spread of the TD error"); ax[2].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
print(f"best episode        {max(scores):8.2f}")
print(f"mean of last 100    {np.mean(scores[-100:]):8.2f}")
print(f"episodes past the flag (score > 90): {sum(s > 90 for s in scores)} of {len(scores)}")
with torch.no_grad():
    _, log_std = agent.actor(torch.tensor([[-0.5, 0.0]]))
    v = agent.critic(torch.tensor([[-0.5, 0.0]]))
print(f"final policy std at the valley floor: {float(torch.exp(log_std)):.4f}")
print(f"critic's value of the valley floor:   {float(v):+.3f}")

## Measuring the variance claim

The unit says the critic reduces the variance of the policy gradient. The two
estimators weight $\nabla_\theta\log\pi$ by different quantities --
$G_k$ for REINFORCE, $\delta_k$ for actor-critic -- so the comparison to make
is between the spreads of those two weights on the *same* trajectories.

The cell below collects both along a batch of episodes under a fixed policy,
which is the honest version of the comparison: with the policies free to
differ, any difference in spread could just be a difference in behaviour.

In [ ]:
COMPARE_EPISODES = 20

seed_all(123)
probe_env = make_env(cfg, seed=123)
probe = pg.ActorCritic(state_size, action_size, cfg)
probe.actor.load_state_dict(agent.actor.state_dict())      # the trained policy
probe.critic.load_state_dict(agent.critic.state_dict())    # and the trained critic

G_all, delta_all = [], []
for _ in range(COMPARE_EPISODES):
    state, _ = probe_env.reset()
    rewards, states, next_states, dones = [], [], [], []
    for _ in range(cfg.max_steps):
        action, _ = probe.select_action(state)
        nxt, r, term, trunc, _ = probe_env.step(action)
        states.append(state); next_states.append(nxt)
        rewards.append(r); dones.append(term or trunc)
        state = nxt
        if term or trunc:
            break
    # Monte-Carlo returns along this episode
    G, acc = [], 0.0
    for r in reversed(rewards):
        acc = r + cfg.gamma * acc
        G.insert(0, acc)
    G_all.extend(G)
    # TD errors along the same episode, under the trained critic
    with torch.no_grad():
        s_t = torch.tensor(np.array(states, dtype=np.float32))
        n_t = torch.tensor(np.array(next_states, dtype=np.float32))
        v = probe.critic(s_t).squeeze(-1)
        vn = probe.critic(n_t).squeeze(-1)
        vn[torch.tensor(dones)] = 0.0
        d = torch.tensor(rewards, dtype=torch.float32) + cfg.gamma * vn - v
    delta_all.extend(d.tolist())

G_all, delta_all = np.array(G_all), np.array(delta_all)
print(f"over {COMPARE_EPISODES} episodes, {len(G_all)} steps, under one fixed policy:\n")
print(f"{'weight on grad log pi':<32}{'mean':>12}{'std':>12}")
print("-" * 56)
print(f"{'Monte-Carlo return G_k':<32}{G_all.mean():>12.4f}{G_all.std():>12.4f}")
print(f"{'TD error delta_k':<32}{delta_all.mean():>12.4f}{delta_all.std():>12.4f}")
print(f"\nratio of standard deviations: {G_all.std() / (delta_all.std() + 1e-12):.1f}x")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(G_all, bins=60, color="C0", alpha=0.8)
ax[0].set_title(r"Monte-Carlo return $G_k$ (REINFORCE's weight)")
ax[0].set_xlabel("value"); ax[0].grid(True, alpha=0.3)
ax[1].hist(delta_all, bins=60, color="C3", alpha=0.8)
ax[1].set_title(r"TD error $\delta_k$ (actor-critic's weight)")
ax[1].set_xlabel("value"); ax[1].grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

The two histograms are the content of the unit's claim. The return
$G_k$ accumulates every reward from step $k$ to the end of the episode, so its
spread grows with the horizon; the TD error $\delta_k$ involves one reward and
two evaluations of the critic, so its spread does not. That is the variance
reduction, and it is structural rather than incidental.

Two honest caveats on the comparison:

- $\delta_k$ is **biased** while $G_k$ is not, exactly as in Unit 11. A
  smaller spread around a slightly wrong centre is the trade, not a free win.
- $\delta_k$'s spread depends on how good the critic is. Early in training,
  when $\hat J$ is nearly arbitrary, the TD error is not much better behaved
  than the return.

## What this does *not* fix

The score almost certainly did not reach $+90$, and the critic did not help
with that. The reason is the one the REINFORCE notebook sets out: on this
environment acting costs $-0.1u^2$ per step and the $+100$ arrives only at the
flag, so until the car has reached the flag once the objective genuinely
prefers doing nothing. The policy's standard deviation shrinks, exploration
stops, and the run settles into that local optimum.

A critic reduces the **variance of the gradient estimate**. It does not change
**what the gradient points at**. The trap here is in the objective, and the
remedies are the ones on the deck's variants slide -- an entropy bonus to keep
the policy wide (Soft Actor-Critic), or a different exploration mechanism
altogether.

That is worth stating plainly because the example is otherwise easy to
misread. It is a good demonstration of the actor-critic *machinery* and of the
variance claim, and a good demonstration of the limits of policy gradient
methods on sparse-reward problems. It is not a demonstration that the method
solves the mountain car.

In [ ]:
mean, std = evaluate(agent, env, episodes=10)
print(f"trained policy over 10 episodes: {mean:.2f} +/- {std:.2f}")
print("(the policy is still stochastic, so this is the behaviour policy's score)")
env.close(); probe_env.close()